#### Layer 1 — Establish the paradox

*Does higher spending actually produce better clinical outcomes?*

*What share of hospitals are high-spend with poor outcomes vs. low-spend with good outcomes?*

In [5]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../cms_hospital.db")


In [7]:
query = """
WITH outcome_scores AS (
    SELECT
        r.facility_id,
        r.facility_name,
        r.state,

        -- MORTALITY (6 measures)
        (CASE WHEN r.MORT_30_AMI_compared  LIKE '%Better%' THEN 1 WHEN r.MORT_30_AMI_compared  LIKE '%Worse%' THEN -1 ELSE 0 END +
         CASE WHEN r.MORT_30_CABG_compared LIKE '%Better%' THEN 1 WHEN r.MORT_30_CABG_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
         CASE WHEN r.MORT_30_COPD_compared LIKE '%Better%' THEN 1 WHEN r.MORT_30_COPD_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
         CASE WHEN r.MORT_30_HF_compared   LIKE '%Better%' THEN 1 WHEN r.MORT_30_HF_compared   LIKE '%Worse%' THEN -1 ELSE 0 END +
         CASE WHEN r.MORT_30_PN_compared   LIKE '%Better%' THEN 1 WHEN r.MORT_30_PN_compared   LIKE '%Worse%' THEN -1 ELSE 0 END +
         CASE WHEN r.MORT_30_STK_compared  LIKE '%Better%' THEN 1 WHEN r.MORT_30_STK_compared  LIKE '%Worse%' THEN -1 ELSE 0 END)
        AS mortality_score,

        -- PATIENT SAFETY / PSI (8 measures)
        (CASE WHEN r.PSI_03_compared LIKE '%Better%' THEN 1 WHEN r.PSI_03_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
         CASE WHEN r.PSI_04_compared LIKE '%Better%' THEN 1 WHEN r.PSI_04_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
         CASE WHEN r.PSI_06_compared LIKE '%Better%' THEN 1 WHEN r.PSI_06_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
         CASE WHEN r.PSI_08_compared LIKE '%Better%' THEN 1 WHEN r.PSI_08_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
         CASE WHEN r.PSI_09_compared LIKE '%Better%' THEN 1 WHEN r.PSI_09_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
         CASE WHEN r.PSI_11_compared LIKE '%Better%' THEN 1 WHEN r.PSI_11_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
         CASE WHEN r.PSI_12_compared LIKE '%Better%' THEN 1 WHEN r.PSI_12_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
         CASE WHEN r.PSI_90_compared LIKE '%Better%' THEN 1 WHEN r.PSI_90_compared LIKE '%Worse%' THEN -1 ELSE 0 END)
        AS psi_score,

        -- READMISSIONS (2 measures)
        (CASE WHEN r.COMP_HIP_KNEE_compared LIKE '%Better%' THEN 1 WHEN r.COMP_HIP_KNEE_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
         CASE WHEN r.Hybrid_HWM_compared    LIKE '%Better%' THEN 1 WHEN r.Hybrid_HWM_compared    LIKE '%Worse%' THEN -1 ELSE 0 END)
        AS readmission_score,

        -- INFECTIONS / HAI (6 SIR measures, benchmark = 1.0)
        (CASE WHEN i.HAI_1_SIR < 1 THEN 1 WHEN i.HAI_1_SIR > 1 THEN -1 ELSE 0 END +
         CASE WHEN i.HAI_2_SIR < 1 THEN 1 WHEN i.HAI_2_SIR > 1 THEN -1 ELSE 0 END +
         CASE WHEN i.HAI_3_SIR < 1 THEN 1 WHEN i.HAI_3_SIR > 1 THEN -1 ELSE 0 END +
         CASE WHEN i.HAI_4_SIR < 1 THEN 1 WHEN i.HAI_4_SIR > 1 THEN -1 ELSE 0 END +
         CASE WHEN i.HAI_5_SIR < 1 THEN 1 WHEN i.HAI_5_SIR > 1 THEN -1 ELSE 0 END +
         CASE WHEN i.HAI_6_SIR < 1 THEN 1 WHEN i.HAI_6_SIR > 1 THEN -1 ELSE 0 END)
        AS hai_score,

        -- PATIENT EXPERIENCE (HCAHPS star rating)
        CASE
            WHEN h.H_STAR_RATING >= 4 THEN 1
            WHEN h.H_STAR_RATING <= 2 THEN -1
            ELSE 0
        END AS hcahps_score

    FROM readmissions_and_deaths r
    LEFT JOIN healthcare_infections  i ON r.facility_id = i.facility_id
    LEFT JOIN hcahps_patient_survey  h ON r.facility_id = h.facility_id
),

final AS (
    SELECT
        o.facility_id,
        o.facility_name,
        o.state,
        (o.mortality_score + o.psi_score + o.readmission_score + o.hai_score + o.hcahps_score) AS total_outcome_score,
        o.mortality_score,
        o.psi_score,
        o.readmission_score,
        o.hai_score,
        o.hcahps_score,
        s.complete_total,
        CASE WHEN s.complete_total > (SELECT AVG(complete_total) FROM medicare_spending)
             THEN 'High Spend' ELSE 'Low Spend' END AS spend_category
    FROM outcome_scores o
    LEFT JOIN medicare_spending s ON o.facility_id = s.facility_id
    WHERE s.complete_total IS NOT NULL
)

SELECT *
FROM final
ORDER BY total_outcome_score DESC
"""

df_layer1 = pd.read_sql_query(query, conn)
df_layer1

,facility_id,facility_name,state,total_outcome_score,mortality_score,psi_score,readmission_score,hai_score,hcahps_score,complete_total,spend_category
0,330101,NEW YORK-PRESBYTERIAN HOSPITAL,NY,18,5,6,1,6,0,30029,High Spend
1,330214,NYU LANGONE HOSPITALS,NY,16,6,4,2,4,0,29515,High Spend
2,140010,NORTHSHORE UNIVERSITY HEALTHSYSTEM - EVANSTON ...,IL,15,5,4,2,4,0,26776,High Spend
3,220071,MASSACHUSETTS GENERAL HOSPITAL,MA,15,5,2,1,6,1,31948,High Spend
4,450358,HOUSTON METHODIST HOSPITAL,TX,15,3,4,1,6,1,30838,High Spend
...,...,...,...,...,...,...,...,...,...,...,...
2887,260001,MERCY HOSPITAL JOPLIN,MO,-5,-2,0,0,-3,0,25559,Low Spend
2888,360218,LICKING MEMORIAL HOSPITAL,OH,-5,0,0,0,-5,0,24475,Low Spend
2889,510022,CHARLESTON AREA MEDICAL CENTER,WV,-5,0,-3,0,-2,0,27262,High Spend
2890,050045,EL CENTRO REGIONAL MEDICAL CENTER,CA,-6,-1,0,-1,-3,-1,24306,Low Spend


## Note — How to Interpret the Total Outcome Score

A high `total_outcome_score` means the hospital is **consistently beating 
the national benchmark** across multiple clinical dimensions simultaneously.
A low or negative score means the opposite.

**Important: 0 is not bad — 0 means average.**

| Score Range | Interpretation |
|---|---|
| +10 to +23 | Genuinely exceptional — outperforming nationally across most measures |
| -3 to +3 | Average — performing as expected, nothing alarming |
| -10 to -23 | Systematically underperforming across multiple dimensions |

**The hospitals that answer our central research question are the ones with 
high negative scores but high spending.**
That is a hospital that spends above average AND still performs worse than 
the national rate on mortality, safety, readmissions, and infections — 
not despite the spending, but alongside it.
That is not just a paradox. That is a policy problem.

In [8]:
df_layer1['outcome_category'] = df_layer1['total_outcome_score'].apply(
    lambda x: 'Good Outcome' if x > 0 else ('Poor Outcome' if x < 0 else 'Average')
)

quadrants = df_layer1.groupby(['spend_category', 'outcome_category']).size().reset_index(name='hospital_count')
total = len(df_layer1)
quadrants['percentage'] = (quadrants['hospital_count'] / total * 100).round(1)

print(quadrants.sort_values(['spend_category', 'outcome_category']))

  spend_category outcome_category  hospital_count  percentage
0     High Spend          Average             141         4.9
1     High Spend     Good Outcome            1201        41.5
2     High Spend     Poor Outcome              99         3.4
3      Low Spend          Average             276         9.5
4      Low Spend     Good Outcome            1073        37.1
5      Low Spend     Poor Outcome             102         3.5


In [10]:
correlation = df_layer1['complete_total'].corr(df_layer1['total_outcome_score'])
print(f"Correlation between spending and outcome score: {correlation:.4f}")

if abs(correlation) < 0.2:
    print(" Very weak relationship. Spending does NOT predict outcomes.")
elif correlation < 0:
    print(" Negative relationship. Higher spending associates with worse outcomes.")
else:
    print("Positive but weak. Spending slightly associates with better outcomes.")

Correlation between spending and outcome score: 0.2369
Positive but weak. Spending slightly associates with better outcomes.


## Finding 1 - Correlation: Spending vs Outcome Score

**Correlation coefficient: 0.2369**

The relationship between hospital spending and clinical outcomes is **weak and positive**.
This means:
- Higher spending is *slightly* associated with better outcomes
- But spending explains only a small fraction of why some hospitals perform better
- The majority of outcome variation has **nothing to do with how much is spent**

A correlation of 0.24 in a system this complex is practically saying:
> "You can throw more money at a hospital and it *might* get marginally better —
> but money is not what separates good hospitals from bad ones."

This sets up the core question for the rest of this analysis:
**If spending doesn't drive outcomes, what does?**

In [15]:
query = """
    WITH scored AS (
        SELECT
            s.facility_id,
            s.facility_name,
            s.state,
            s.complete_total,
            CASE WHEN r.MORT_30_AMI_compared  LIKE '%Better%' THEN 1 WHEN r.MORT_30_AMI_compared  LIKE '%Worse%' THEN -1 ELSE 0 END +
            CASE WHEN r.MORT_30_CABG_compared LIKE '%Better%' THEN 1 WHEN r.MORT_30_CABG_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
            CASE WHEN r.MORT_30_COPD_compared LIKE '%Better%' THEN 1 WHEN r.MORT_30_COPD_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
            CASE WHEN r.MORT_30_HF_compared   LIKE '%Better%' THEN 1 WHEN r.MORT_30_HF_compared   LIKE '%Worse%' THEN -1 ELSE 0 END +
            CASE WHEN r.MORT_30_PN_compared   LIKE '%Better%' THEN 1 WHEN r.MORT_30_PN_compared   LIKE '%Worse%' THEN -1 ELSE 0 END +
            CASE WHEN r.MORT_30_STK_compared  LIKE '%Better%' THEN 1 WHEN r.MORT_30_STK_compared  LIKE '%Worse%' THEN -1 ELSE 0 END
            AS mortality_score,

            CASE WHEN r.PSI_03_compared LIKE '%Better%' THEN 1 WHEN r.PSI_03_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
            CASE WHEN r.PSI_04_compared LIKE '%Better%' THEN 1 WHEN r.PSI_04_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
            CASE WHEN r.PSI_06_compared LIKE '%Better%' THEN 1 WHEN r.PSI_06_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
            CASE WHEN r.PSI_08_compared LIKE '%Better%' THEN 1 WHEN r.PSI_08_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
            CASE WHEN r.PSI_09_compared LIKE '%Better%' THEN 1 WHEN r.PSI_09_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
            CASE WHEN r.PSI_11_compared LIKE '%Better%' THEN 1 WHEN r.PSI_11_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
            CASE WHEN r.PSI_12_compared LIKE '%Better%' THEN 1 WHEN r.PSI_12_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
            CASE WHEN r.PSI_90_compared LIKE '%Better%' THEN 1 WHEN r.PSI_90_compared LIKE '%Worse%' THEN -1 ELSE 0 END
            AS psi_score,

            CASE WHEN r.COMP_HIP_KNEE_compared LIKE '%Better%' THEN 1 WHEN r.COMP_HIP_KNEE_compared LIKE '%Worse%' THEN -1 ELSE 0 END +
            CASE WHEN r.Hybrid_HWM_compared    LIKE '%Better%' THEN 1 WHEN r.Hybrid_HWM_compared    LIKE '%Worse%' THEN -1 ELSE 0 END
            AS readmission_score,

            CASE WHEN i.HAI_1_SIR < 1 THEN 1 WHEN i.HAI_1_SIR > 1 THEN -1 ELSE 0 END +
            CASE WHEN i.HAI_2_SIR < 1 THEN 1 WHEN i.HAI_2_SIR > 1 THEN -1 ELSE 0 END +
            CASE WHEN i.HAI_3_SIR < 1 THEN 1 WHEN i.HAI_3_SIR > 1 THEN -1 ELSE 0 END +
            CASE WHEN i.HAI_4_SIR < 1 THEN 1 WHEN i.HAI_4_SIR > 1 THEN -1 ELSE 0 END +
            CASE WHEN i.HAI_5_SIR < 1 THEN 1 WHEN i.HAI_5_SIR > 1 THEN -1 ELSE 0 END +
            CASE WHEN i.HAI_6_SIR < 1 THEN 1 WHEN i.HAI_6_SIR > 1 THEN -1 ELSE 0 END
            AS hai_score,

            CASE WHEN h.H_STAR_RATING >= 4 THEN 1 WHEN h.H_STAR_RATING <= 2 THEN -1 ELSE 0 END
            AS hcahps_score

        FROM medicare_spending s
        LEFT JOIN readmissions_and_deaths r ON s.facility_id = r.facility_id
        LEFT JOIN healthcare_infections   i ON s.facility_id = i.facility_id
        LEFT JOIN hcahps_patient_survey   h ON s.facility_id = h.facility_id
        WHERE s.complete_total IS NOT NULL
    )

    SELECT
        facility_id,
        facility_name,
        state,
        complete_total,
        (mortality_score + psi_score + readmission_score + hai_score + hcahps_score) AS total_outcome_score,
        mortality_score,
        psi_score,
        readmission_score,
        hai_score,
        hcahps_score
    FROM scored
    WHERE complete_total > (SELECT AVG(complete_total) FROM medicare_spending)
      AND (mortality_score + psi_score + readmission_score + hai_score + hcahps_score) < 0
    ORDER BY (mortality_score + psi_score + readmission_score + hai_score + hcahps_score) ASC

"""
paradox_hospitals = pd.read_sql_query(query, conn)
paradox_hospitals


,facility_id,facility_name,state,complete_total,total_outcome_score,mortality_score,psi_score,readmission_score,hai_score,hcahps_score
0,110034,WELLSTAR MCG HEALTH AFFILIATED WITH MED COL,GA,30606,-6,-1,-2,0,-2,-1
1,510022,CHARLESTON AREA MEDICAL CENTER,WV,27262,-5,0,-3,0,-2,0
2,40020,ST BERNARDS MEDICAL CENTER,AR,28077,-4,-3,-2,0,1,0
3,140007,SAINT JOSEPH MEDICAL CENTER,IL,27337,-4,0,-2,0,-1,-1
4,180104,BAPTIST HEALTH PADUCAH,KY,26225,-4,0,-2,0,-3,1
...,...,...,...,...,...,...,...,...,...,...
94,450147,DE TAR HOSPITAL NAVARRO,TX,29316,-1,0,0,0,-1,0
95,450162,GRACE SURGICAL HOSPITAL,TX,28979,-1,0,0,0,-1,0
96,450222,HCA HOUSTON HEALTHCARE CONROE,TX,30695,-1,0,-1,0,1,-1
97,450713,ST DAVID'S SOUTH AUSTIN MEDICAL CENTER,TX,29746,-1,-1,-1,0,1,0


## Paradox Hospitals - High Spend, Poor Outcome

This table contains **99 hospitals** that simultaneously:
- Spend **above the national average** on Medicare per episode (`complete_total`)
- Score **below 0** on the composite clinical outcome score

A total outcome score below 0 does not mean a hospital fails on every dimension.
It means the hospital has **more negative signals than positive ones** across the
23 CMS benchmarks — at least one dimension is dragging the overall score down,
and nothing else is compensating for it.

The 0s you see in individual dimension columns are not failures —
they mean that hospital is performing exactly at the national average on that measure.
The failure is visible in the negative columns: that is where the money
is not translating into quality.

These 99 hospitals are the core of our research question.
They spend more. They perform worse. The next layers of this analysis
will investigate **why.**